# Saccade Collection Pipeline

**Environment:** Activate the `eye_repo` conda environment and install the package (`pip install -e .`) before running this notebook.

This notebook curates saccade events across animals and recording blocks, compatible with the current **eye_tracking_system_tools** repository. It produces `all_saccade_collection` (a DataFrame of saccades with animal, block, eye, timestamps, and metrics) for downstream use (filtering by type, LFP extraction via `oe_rec.get_data()`, etc.).

**Prerequisites:**
- Block synchronization completed (`block_synchronization.ipynb`).
- `left_eye_data.csv` and `right_eye_data.csv` in each block's `analysis/` folder.
- `final_sync_df.csv` (or `blocksync_df.csv`) in each block's `analysis/` folder.

**Workflow:**
1. Build `block_collection` and `block_dict` from config (animals, block lists).
2. Load `final_sync_df` and eye data per block.
3. Run saccade detection per eye, attach `animal` / `block` / `eye`.
4. Split into synced (binocular) vs non-synced (monocular) events.
5. Build `all_saccade_collection = synced ∪ non_synced`.

*Electrophysiology (LFP) extraction will be added in a later step.*

In [1]:
# =============================================================================
# IMPORTS
# =============================================================================
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
import tqdm

from eye_tracking_system_tools.preprocessing import BlockSync
from eye_tracking_system_tools.preprocessing import utility_functions as uf

## Config

Set `experiment_path` (parent of animal folders), `animals`, `block_lists` (one list of block numbers per animal), and optional `bad_blocks`. Adjust `speed_threshold` and `diff_threshold` (ms) for saccade detection and L/R pairing.

In [2]:
experiment_path = Path(r"D:\sample_data_for_eye_repo")  # parent of animal folders
animals = ["PV_106"]
block_lists = [[15]]  # one list of block numbers per animal
bad_blocks = []
speed_threshold = 2.0   # saccade detection (speed_r > threshold)
diff_threshold_ms = 680  # max |t_L - t_R| (ms) to count as binocular pair

# Export: where to save all_saccade_collection (general analysis folder)
export_dir = experiment_path / "analysis" / "saccade_collections"
export_filename = "all_saccade_collection.csv"

## Helpers: block collection, load sync & eye data

- `create_block_collections`: builds `block_collection` (list of BlockSync) and `block_dict` with keys `"{animal}_block_{block_num}"`.
- `load_final_sync_df`: loads `final_sync_df.csv` or `blocksync_df.csv` into `block.final_sync_df` / `block.blocksync_df`.
- `load_eye_data`: loads `left_eye_data.csv` and `right_eye_data.csv` into `block.left_eye_data` / `block.right_eye_data`. Expects `OE_timestamp`, `center_x`, `center_y`, `ms_axis` (and optionally `pupil_diameter`).

In [3]:
def create_block_collections(animals, block_lists, experiment_path, bad_blocks=None):
    """Build block_collection and block_dict from animals and block lists."""
    if bad_blocks is None:
        bad_blocks = []
    block_collection = []
    block_dict = {}
    for animal, blocks in zip(animals, block_lists):
        current = uf.block_generator(
            block_numbers=blocks,
            experiment_path=experiment_path,
            animal=animal,
            bad_blocks=bad_blocks,

        )
        block_collection.extend(current)
        for b in current:
            block_dict[f"{animal}_block_{b.block_num}"] = b
    return block_collection, block_dict


def load_final_sync_df(block, filename=None, verbose=True):
    """Load final_sync_df from block.analysis_path and set block.final_sync_df / block.blocksync_df."""
    ap = Path(block.analysis_path)
    candidates = [filename] if filename else ["final_sync_df.csv", "blocksync_df.csv"]
    path = None
    for name in candidates:
        p = ap / name
        if p.exists():
            path = p
            break
    if path is None:
        raise FileNotFoundError(f"No sync file in {ap}. Tried: {candidates}")
    df = pd.read_csv(path)
    required = ["Arena_TTL", "Arena_frame", "L_eye_frame", "R_eye_frame", "L_values", "R_values"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{path.name} missing columns: {missing}")
    df = df.copy()
    df["Arena_TTL"] = df["Arena_TTL"].astype(float)
    if "ms_axis" not in df.columns and hasattr(block, "sample_rate") and block.sample_rate:
        df["ms_axis"] = df["Arena_TTL"] / (block.sample_rate / 1000)
    block.final_sync_df = df
    block.blocksync_df = df
    if verbose:
        print(f"[OK] Loaded {path.name} -> block.final_sync_df (rows={len(df):,})")
    return df


def load_eye_data(block, verbose=True):
    """Load left/right_eye_data from block.analysis_path. Expects OE_timestamp, center_x, center_y, ms_axis."""
    ap = Path(block.analysis_path)
    lp = ap / "left_eye_data.csv"
    rp = ap / "right_eye_data.csv"
    if not lp.exists() or not rp.exists():
        raise FileNotFoundError(f"Eye data not found in {ap}. Run block_synchronization pipeline first.")
    block.left_eye_data = pd.read_csv(lp, index_col=0, engine="python")
    block.right_eye_data = pd.read_csv(rp, index_col=0, engine="python")
    for name, df in [("left", block.left_eye_data), ("right", block.right_eye_data)]:
        for c in ["OE_timestamp", "center_x", "center_y", "ms_axis"]:
            if c not in df.columns:
                raise ValueError(f"{name}_eye_data missing column: {c}")
    if verbose:
        print(f"[OK] Loaded eye data for block {block.block_num} (L={len(block.left_eye_data)}, R={len(block.right_eye_data)})")

## Build block collection and load data

Instantiate blocks, load `final_sync_df` and eye data per block. Ensure each block has `sample_rate` (for `ms_axis`); it is set by BlockSync from the Open Ephys settings.

In [4]:
block_collection, block_dict = create_block_collections(
    animals=animals,
    block_lists=block_lists,
    experiment_path=experiment_path,
    bad_blocks=bad_blocks,

)
for block in block_collection:
    load_final_sync_df(block)
    load_eye_data(block)
print(f"Blocks: {list(block_dict.keys())}")

instantiated block number 015 at Path: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015, new OE version
Found the sample rate for block 015 in the xml file, it is 20000 Hz

Extracting meta data from: D:\sample_data_for_eye_repo\PV_106\2025_09_04\block_015\oe_files\PV106_IMU_trial4_prey_2025-09-04_13-24-17\Record Node 106...
Analog channel numbers contain duplicates!!! Reordering numbers serially.

Extracting time stamp information...

Error!!! Some blocks are missing in recording!!!

Checking integrity of all records in ch1...

Metadata extraction complete.
created the .oe_rec attribute as an open ephys recording obj with get_data functionality (standalone mode, no metadata file)
retrieving zertoh sample number for block 015
got it!
[OK] Loaded final_sync_df.csv -> block.final_sync_df (rows=23,091)
[OK] Loaded eye data for block 015 (L=11949, R=11949)
Blocks: ['PV_106_block_015']


## Saccade detection

`create_saccade_events_df` detects saccades from eye data (speed_r > threshold), computes onset/offset, magnitude, angle, and optional speed/diameter profiles. Uses `center_x`, `center_y`, `ms_axis`, `OE_timestamp`; `pupil_diameter` is optional.

In [5]:
def create_saccade_events_df(eye_data_df, speed_threshold, magnitude_calib=1.0, speed_profile=True, use_pupil_diameter=True):
    """
    Detect saccade events from eye tracking data (speed_r > threshold).
    Returns (df_with_speed, saccade_events_df). df is a copy with speed cols; original unchanged.
    """
    df = eye_data_df.copy()
    df["speed_x"] = df["center_x"].diff()
    df["speed_y"] = df["center_y"].diff()
    df["speed_r"] = (df["speed_x"] ** 2 + df["speed_y"] ** 2) ** 0.5
    df["is_saccade"] = df["speed_r"] > speed_threshold

    on_off = df["is_saccade"].astype(int) - df["is_saccade"].shift(periods=1, fill_value=False).astype(int)
    on_inds = np.where(on_off == 1)[0] - 1
    off_inds = np.where(on_off == -1)[0]
    if len(on_inds) == 0 or len(off_inds) == 0:
        ev = pd.DataFrame(columns=[
            "saccade_start_ind", "saccade_end_ind", "saccade_start_timestamp", "saccade_end_timestamp",
            "saccade_on_ms", "saccade_off_ms", "length", "magnitude_raw", "magnitude", "angle",
            "initial_x", "initial_y", "end_x", "end_y", "calib_dx", "calib_dy",
        ])
        if speed_profile:
            ev["speed_profile"] = []
        if use_pupil_diameter and "pupil_diameter" in df.columns:
            ev["diameter_profile"] = []
        df = df.drop(columns=["speed_x", "speed_y", "speed_r", "is_saccade"], errors="ignore")
        return df, ev

    on_ms = df["ms_axis"].iloc[on_inds].values
    on_ts = df["OE_timestamp"].iloc[on_inds].values
    off_ts = df["OE_timestamp"].iloc[off_inds].values
    off_ms = df["ms_axis"].iloc[off_inds].values

    ev = pd.DataFrame({
        "saccade_start_ind": on_inds,
        "saccade_end_ind": off_inds,
        "saccade_start_timestamp": on_ts,
        "saccade_end_timestamp": off_ts,
        "saccade_on_ms": on_ms,
        "saccade_off_ms": off_ms,
    })
    ev["length"] = ev["saccade_end_ind"] - ev["saccade_start_ind"]

    distances, angles, speed_list, diameter_list = [], [], [], []
    for _, row in tqdm.tqdm(ev.iterrows(), total=len(ev), desc="saccade metrics"):
        seg = df.loc[(df["OE_timestamp"] >= row["saccade_start_timestamp"]) &
                     (df["OE_timestamp"] <= row["saccade_end_timestamp"])]
        dist = seg["speed_r"].sum()
        distances.append(dist)
        if speed_profile:
            speed_list.append(seg["speed_r"].values)
        if use_pupil_diameter and "pupil_diameter" in df.columns:
            diameter_list.append(seg["pupil_diameter"].values)
        elif use_pupil_diameter:
            diameter_list.append(np.full(len(seg), np.nan))
        xi, yi = seg.iloc[0][["center_x", "center_y"]]
        xe, ye = seg.iloc[-1][["center_x", "center_y"]]
        ang = np.arctan2(ye - yi, xe - xi)
        angles.append(ang)

    ev["magnitude_raw"] = np.array(distances)
    ev["magnitude"] = np.array(distances) * magnitude_calib
    ev["angle"] = np.where(np.isnan(angles), np.nan, np.rad2deg(angles) % 360)
    start_ts = ev["saccade_start_timestamp"].values
    end_ts = ev["saccade_end_timestamp"].values
    start_df = df[df["OE_timestamp"].isin(start_ts)]
    end_df = df[df["OE_timestamp"].isin(end_ts)]
    ev["initial_x"] = start_df["center_x"].values
    ev["initial_y"] = start_df["center_y"].values
    ev["end_x"] = end_df["center_x"].values
    ev["end_y"] = end_df["center_y"].values
    ev["calib_dx"] = (ev["end_x"].values - ev["initial_x"].values) * magnitude_calib
    ev["calib_dy"] = (ev["end_y"].values - ev["initial_y"].values) * magnitude_calib
    if speed_profile:
        ev["speed_profile"] = speed_list
    if use_pupil_diameter and (("pupil_diameter" in df.columns) or diameter_list):
        ev["diameter_profile"] = diameter_list

    df = df.drop(columns=["speed_x", "speed_y", "speed_r", "is_saccade"], errors="ignore")
    return df, ev

## Per-block saccade extraction and collection

Run saccade detection on left and right eye data for each block, add `animal`, `block`, `eye`, then concatenate into `saccade_df_list` (one DataFrame per block, L+R stacked).

In [6]:
saccade_df_list = []
for block in block_collection:
    animal = block.animal_call
    blk = str(block.block_num)
    out = []
    for eye, attr in [("L", "left_eye_data"), ("R", "right_eye_data")]:
        eye_df = getattr(block, attr)
        _, sacc_ev = create_saccade_events_df(
            eye_df, speed_threshold,
            magnitude_calib=1.0, speed_profile=True, use_pupil_diameter="pupil_diameter" in eye_df.columns,
        )
        sacc_ev["animal"] = animal
        sacc_ev["block"] = blk
        sacc_ev["eye"] = eye
        out.append(sacc_ev)
    combined = pd.concat(out, ignore_index=True)
    saccade_df_list.append(combined)
saccade_collection = pd.concat(saccade_df_list, ignore_index=True)
print(f"Saccades per block: {[len(d) for d in saccade_df_list]}; total {len(saccade_collection)}")

saccade metrics: 100%|██████████| 210/210 [00:00<00:00, 1473.52it/s]

Saccades per block: [443]; total 443


## Synced vs non-synced (binocular vs monocular)

`find_synced_saccades`: within each block, pair L and R saccades by `saccade_on_ms` (within `diff_threshold_ms`). Paired → synced (binocular); unpaired → non_synced (monocular). `combine_synced_dataframes` builds a single `synced_saccade_collection` with `Main`/`Sub` indices across blocks.

In [7]:
def find_synced_saccades(df, diff_threshold_ms=680, on_col="saccade_on_ms"):
    """Split saccades into synced (L–R pairs) and non_synced (unpaired). df has both eyes, animal, block."""
    l_df = df.query('eye == "L"').copy()
    r_df = df.query('eye == "R"').copy()
    synced_rows = []
    non_synced_rows = []

    for _, row in l_df.iterrows():
        t_l = row[on_col]
        dt = np.abs(r_df[on_col].values - t_l)
        ind = np.argmin(dt)
        if dt[ind] < diff_threshold_ms:
            synced_rows.append((row, r_df.iloc[ind]))
        else:
            non_synced_rows.append(row)

    r_matched = r_df.index.isin([r.index for _, r in synced_rows])
    r_leftovers = r_df.loc[~r_matched]

    n = len(synced_rows)
    idx = pd.MultiIndex.from_tuples(
        [(i, "L") for i in range(n)] + [(i, "R") for i in range(n)], names=["Main", "Sub"]
    )
    synced_df = pd.DataFrame(index=idx, columns=df.columns)
    for i, (l_row, r_row) in enumerate(synced_rows):
        synced_df.loc[(i, "L")] = l_row
        synced_df.loc[(i, "R")] = r_row

    non_synced_df = pd.concat([
        pd.DataFrame(non_synced_rows, columns=df.columns),
        r_leftovers,
    ], ignore_index=True)
    return synced_df, non_synced_df


def combine_synced_dataframes(synced_df_list):
    """Concatenate per-block synced DFs, reindex Main across blocks, reset_index."""
    out = []
    start = 0
    for sdf in synced_df_list:
        n = len(sdf) // 2
        if n == 0:
            continue
        idx = pd.MultiIndex.from_tuples(
            [(start + i, "L") for i in range(n)] + [(start + i, "R") for i in range(n)],
            names=["Main", "Sub"],
        )
        sdf = sdf.set_axis(idx)
        out.append(sdf)
        start += n
    if not out:
        return pd.DataFrame()
    combined = pd.concat(out)
    return combined.reset_index()

## Build `synced_saccade_collection`, `non_synced_saccade_collection`, `all_saccade_collection`

Run `find_synced_saccades` on each block's saccade DataFrame, then combine and concatenate.

In [8]:
synced_df_list = []
non_synced_df_list = []
for saccade_df in saccade_df_list:
    synced_df, non_synced_df = find_synced_saccades(
        saccade_df.dropna(subset=["saccade_on_ms"]), diff_threshold_ms=diff_threshold_ms, on_col="saccade_on_ms"
    )
    synced_df_list.append(synced_df)
    non_synced_df_list.append(non_synced_df)

synced_saccade_collection = combine_synced_dataframes(synced_df_list)
non_synced_saccade_collection = pd.concat(non_synced_df_list, ignore_index=True)
all_saccade_collection = pd.concat([synced_saccade_collection, non_synced_saccade_collection], ignore_index=True)

print(f"Synced: {len(synced_saccade_collection)}, non_synced: {len(non_synced_saccade_collection)}, all: {len(all_saccade_collection)}")

Synced: 452, non_synced: 217, all: 669


## Export `all_saccade_collection`

Save `all_saccade_collection` to `export_dir` (general analysis folder). Optionally save a small manifest (animals, blocks, counts) as JSON for downstream pipelines.

In [9]:
import json
from datetime import datetime

export_dir.mkdir(parents=True, exist_ok=True)
path_csv = export_dir / export_filename
all_saccade_collection.to_csv(path_csv, index=False)
print(f"[OK] Exported all_saccade_collection -> {path_csv} (rows={len(all_saccade_collection):,})")

n_synced_pairs = int(all_saccade_collection["Main"].dropna().nunique())
n_synced_rows = int(all_saccade_collection["Main"].notna().sum())
n_non_synced = int(all_saccade_collection["Main"].isna().sum())
manifest = {
    "export_time": datetime.now().isoformat(),
    "path": str(path_csv),
    "animals": sorted(all_saccade_collection["animal"].dropna().unique().astype(str).tolist()),
    "blocks": sorted(all_saccade_collection["block"].dropna().unique().astype(str).tolist()),
    "n_synced_pairs": n_synced_pairs,
    "n_synced_rows": n_synced_rows,
    "n_non_synced": n_non_synced,
    "n_total": len(all_saccade_collection),
}
path_manifest = export_dir / "all_saccade_collection_manifest.json"
with open(path_manifest, "w") as f:
    json.dump(manifest, f, indent=2)
print(f"[OK] Manifest -> {path_manifest}")

[OK] Exported all_saccade_collection -> D:\sample_data_for_eye_repo\analysis\saccade_collections\all_saccade_collection.csv (rows=669)
[OK] Manifest -> D:\sample_data_for_eye_repo\analysis\saccade_collections\all_saccade_collection_manifest.json


## Summary

- **`block_collection`**: list of `BlockSync` objects.
- **`block_dict`**: `"{animal}_block_{block_num}"` → `BlockSync`.
- **`all_saccade_collection`**: DataFrame of all saccades (synced + non_synced) with `animal`, `block`, `eye`, `saccade_on_ms`, `saccade_off_ms`, `magnitude`, `angle`, etc. Synced rows have `Main` / `Sub`; non_synced do not.

Next steps (later): filter by type (e.g. `head_movement`, verified monocular), then LFP extraction via `block.oe_rec.get_data(...)`.

In [10]:
all_saccade_collection.head()

,Main,Sub,saccade_start_ind,saccade_end_ind,saccade_start_timestamp,saccade_end_timestamp,saccade_on_ms,saccade_off_ms,length,magnitude_raw,...,initial_x,initial_y,end_x,end_y,calib_dx,calib_dy,speed_profile,animal,block,eye
0,0.0,L,1103,1107,1185224.0,1186556.0,59261.2,59327.8,4,38.045673,...,273.48647,240.891843,272.964242,229.347818,-0.522228,-11.544025,"[nan, 18.159848009797628, 12.737395292211708, ...",PV_106,015,L
1,1.0,L,1133,1136,1195214.0,1196213.0,59760.7,59810.65,3,10.652197,...,270.868028,229.577565,266.92566,238.489003,-3.942368,8.911438,"[0.6300972280362364, 3.720890957119532, 5.3045...",PV_106,015,L
2,2.0,L,1168,1171,1206869.0,1207868.0,60343.45,60393.4,3,5.766493,...,271.272991,233.466588,268.03661,237.68319,-3.236381,4.216602,"[0.4192717955807564, 2.2389798124922797, 2.107...",PV_106,015,L
3,3.0,L,1274,1276,1242167.0,1242833.0,62108.35,62141.65,2,7.739014,...,270.229955,237.399945,277.10391,237.792368,6.873955,0.392423,"[0.8304330847227812, 5.241737770711153, 1.6668...",PV_106,015,L
4,4.0,L,1279,1281,1243832.0,1244498.0,62191.6,62224.9,2,4.261767,...,278.31207,238.695116,274.887677,239.564361,-3.424393,0.869245,"[0.547441321731746, 2.1284117892581627, 1.5859...",PV_106,015,L
